In [133]:
import onnxruntime as ort
from embedder import Embedder
import numpy as np
from minsearch import VectorSearch, Index

In [134]:
embedder = Embedder()

## Question 1
The embedder returns a vector of 384 numbers. What's the first value (v[0])?

Answer: **B. -0.02**

In [135]:
query_0_embedded = embedder.encode("How does approximate nearest neighbor search work?")

In [136]:
print(query_0_embedded.shape)

(384,)


In [137]:
print(query_0_embedded[0])

-0.02058203437252893


## Question 2

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

Answer: **0.37**

In [138]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    allowed_extensions={"md"},
    filename_filter=lambda p: not (p.startswith("cohorts"))
)

In [139]:
files = reader.read()

In [140]:
target_path = "02-vector-search/lessons/07-sqlitesearch-vector.md"

In [141]:
md_file = next(
    (f for f in files if f.filename == target_path),
    None,
)

if md_file:
    doc_embedding = embedder.encode(md_file.content)

In [142]:
score = doc_embedding.dot(query_0_embedding)

In [143]:
score

np.float64(0.36107027225589694)

## Question 3

Which file does the highest-scoring chunk belong to (its filename)?

Answer: **C. '02-vector-search/lessons/07-sqlitesearch-vector.md'**

In [144]:
from gitsource import chunk_documents
documents = [f.parse() for f in files]
chunks = chunk_documents(documents, size=2000, step=1000)

In [145]:
contents = [chunk.get("content") for chunk in chunks]
contents_embedded = embedder.encode_batch(contents) 

In [146]:
scores = [query_0_embedding.dot(contents_embedded[i]) for i in range(len(contents_embedded))]
top5 = np.argsort(scores)[:]

In [147]:
index = np.argmax(scores)
chunks[index]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [148]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
[chunks[i]["filename"] for i in top5]

['02-vector-search/lessons/07-sqlitesearch-vector.md',
 '01-agentic-rag/lessons/05-search.md',
 '04-evaluation/lessons/05-search-metrics.md',
 '02-vector-search/lessons/04-vector-search.md',
 '06-best-practices/lessons/03-reranking.md']

## Question 4
Which file is the filename of the first result?

Answer: **B. 04-evaluation/lessons/05-search-metrics.md**

In [149]:
query_1 = "What metric do we use to evaluate a search engine?"
query_1_embedded = embedder.encode(query_1)

In [150]:
vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(contents_embedded, chunks)

In [151]:
results = vector_index.search(query_1_embedded, num_results=5)

In [152]:
[res['filename'] for res in results]

['04-evaluation/lessons/05-search-metrics.md',
 '04-evaluation/lessons/01-intro.md',
 '04-evaluation/README.md',
 '01-agentic-rag/lessons/05-search.md',
 '04-evaluation/lessons/01-intro.md']

## Question 5
Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

Answer: **B. 02-vector-search/lessons/02-embeddings.md**

In [153]:
query_2 = "How do I store vectors in PostgreSQL?"
query_2_embedded = embedder.encode(query_2)

In [154]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)

In [155]:
results_vector_search = vector_index.search(query_2_embedded, num_results=5)

In [156]:
[res['filename'] for res in results_vector_search]

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [157]:
results_full_text_search = index.search(query_2, num_results=5)

In [158]:
[res['filename'] for res in results_full_text_search]

['02-vector-search/lessons/02-embeddings.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

## Question 6
Which file is ranked first after RRF?

Answer: **A. 01-agentic-rag/lessons/01-intro.md**

In [159]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [160]:
query_3 = "How do I give the model access to tools?"
query_3_embedded = embedder.encode(query_3)

In [161]:
results_vector_search = vector_index.search(query_3_embedded, num_results=5)
results_full_text_search = index.search(query_3, num_results=5)

In [162]:
results = rrf([results_vector_search, results_full_text_search])

In [163]:
results[0]['filename']

'01-agentic-rag/lessons/01-intro.md'